In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

model = init_chat_model(
  model="llama-3.1-8b-instant",
  model_provider="GROQ"
)

c:\Users\anand\Desktop\GenAI - Krish Naik\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from pydantic import BaseModel, Field

class Movie(BaseModel):
  title: str = Field(description="Movie Title")
  release_date: int = Field(description="Movie Release Date")
  director: str = Field(description="Movie Director")
  rating: float = Field(description="Movie Rating")

In [3]:
model_with_stroutput = model.with_structured_output(Movie)

In [6]:
model_with_stroutput

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002538DF847A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002538E3FD190>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Movie Title', 'type': 'string'}, 'release_date': {'description': 'Movie Release Date', 'type': 'integer'}, 'director': {'description': 'Movie Director', 'type': 'string'}, 'rating': {'description': 'Movie Rating', 'type': 'number'}}, 'require

In [13]:
model_with_stroutput.invoke("Provide the details about the movie Avengers: Infinity War")

Movie(title='Avengers: Infinity War', release_date=2018, director='Anthony and Joe Russo', rating=8.1)

In [14]:
# Message Output Alongside parsed Structure

class Movie(BaseModel):
  title: str = Field(..., description="Movie Title")
  release_date: int = Field(..., description="Movie Release Date")
  director: str = Field(..., description="Movie Director")
  rating: float = Field(..., description="Movie Rating")
  
model_with_stroutput = model.with_structured_output(Movie, include_raw=True)

response = model_with_stroutput.invoke("Provide the details about the movie Avengers: Infinity War")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9wh876vj6', 'function': {'arguments': '{"director":"Anthony and Joe Russo","rating":8.2,"release_date":2018,"title":"Avengers: Infinity War"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 270, 'total_tokens': 309, 'completion_time': 0.045461931, 'completion_tokens_details': None, 'prompt_time': 0.014941619, 'prompt_tokens_details': None, 'queue_time': 0.158455751, 'total_time': 0.06040355}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec073-7f04-7923-a058-694ed94bb083-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Anthony and Joe Russo', 'rating': 8.2, 'release_date': 2018, 'title': 'Avengers: Infinity War'}, 'id': '9wh876vj6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

In [23]:
# NESTED STRUCTURE

class Actor(BaseModel):
  name: str
  role: str
  
class MovieDetails(BaseModel):
  title: str
  year: int
  cast: list[Actor]
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")
  
model_with_stroutput = model.with_structured_output(MovieDetails)

In [24]:
response = model_with_stroutput.invoke("Provide the movie details for Avengers: Infinity War")

response

MovieDetails(title='Avengers: Infinity War', year=2018, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark/Iron Man'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Mark Ruffalo', role='Bruce Banner/Hulk'), Actor(name='Chris Evans', role='Steve Rogers/Captain America'), Actor(name='Scarlett Johansson', role='Natasha Romanoff/Black Widow'), Actor(name='Benedict Cumberbatch', role='Doctor Strange'), Actor(name='Don Cheadle', role='James Rhodes/War Machine'), Actor(name='Tom Holland', role='Peter Parker/Spider-Man'), Actor(name='Chadwick Boseman', role="T'Challa/Black Panther"), Actor(name='Paul Bettany', role='Vision'), Actor(name='Elizabeth Olsen', role='Wanda Maximoff/Scarlet Witch'), Actor(name='Anthony Mackie', role='Sam Wilson/Falcon'), Actor(name='Sebastian Stan', role='Bucky Barnes/Winter Soldier'), Actor(name='Danai Gurira', role='Okoye'), Actor(name='Letitia Wright', role='Shuri'), Actor(name='Dave Bautista', role='Drax the Destroyer'), Actor(name='Zoe Saldana', ro

#### TypedDict

In [29]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
  title: Annotated[str, ..., "Movie Title"]
  year: Annotated[int, ..., "Movie Release year"]
  director: Annotated[str, ..., "Movie Director"]
  rating: Annotated[float, ..., "Movie Rating"]

model_with_typedict = model.with_structured_output(MovieDict, method="json_mode")

In [31]:
# response = model_with_typedict.invoke("Provide details about the movie The Incredible Hulk")

# response